# ML 工程：保存输入，真实滑动训练，分开消费信号

**阅读时间范围：**第 1–13 节保留此前两折和 v1 TopK 的历史证据与当时消费限制。第 14 节新增 CaseC 的正式 API 验收单元，并列出 CaseA/B 的待接能力。新增单元均未执行，真实输入到位后从第 14 节单独开始；HTML 尚未重新生成。

这一页沿冻结输入、registry Feature、成熟标签、65-session 滑动 Dataset、模型与保存 Signal 展开。**两个真实滑动 fold 已保存并通过独立复审**。另用同一个旧 v1 Signal 的 Top3/Top5 对照说明账户复用；新 v2 Signal 目前仅通过中立验证，不能送入账户。全年和五年尚未运行，收益与参数优劣不作为工程通过条件。

合同归 [Research §4.7](../docs/design/05_axiom_research.md#stock-saved-fold-clock-contract)、[Core](../docs/design/03_axiom_core.md)、[Trade](../docs/design/04_axiom_trade.md)。本页读取 owner 保存证据，不重新执行业务。

## 1. 数据与计算走哪条路

![Axiom ML 架构](../docs/design/ml-engineering-assets/architecture.svg)

实线是已有有界实现；虚线是通用 ModelHandle、schedule、streaming 与恢复目标。Research 用共享后端训练/推理，Core 执行既有 Feature 算子；Engine Runtime 唯一处理账户。**v1 → TopK / Runtime；v2 → 中立校验，账户未准入。**


## 2. 冻结样本：每一份结果属于自己的输入

新主线使用已有固定 Snapshot、314 个 union ID、六个版本化特征和真实交易日历。两个 fit 分别为 2024-02-02、2024-02-08；OOS 为 Feb5–8、Feb19–23。Feature/raw/normalized Label 父件保持原路径、文件 hash、内容 ref 与时钟。

旧 January v1 模型/Signal 与 Top3/Top5 账户是另一组保存结果；不能把新 v2 预测替换进去后沿用旧账户身份。数据可见性仍是 best-effort vendor 证据，不声明历史实时完备性。


In [ ]:
import hashlib
import json
import os
from pathlib import Path


def read_frozen_receipt(path_env, ref_env):
    path = Path(os.environ[path_env])
    raw = path.read_bytes()
    assert "sha256:" + hashlib.sha256(raw).hexdigest() == os.environ[ref_env]
    return json.loads(raw)


receipt = read_frozen_receipt("AXIOM_ML_TEACHING_RECEIPT", "AXIOM_ML_TEACHING_RECEIPT_REF")
fold_receipt = read_frozen_receipt("AXIOM_ML_FOLD_DELIVERY", "AXIOM_ML_FOLD_DELIVERY_REF")
assert receipt["contract_version"] == "private_ml_teaching_acceptance_v1"
assert fold_receipt["status"] == "PASS_OWNER_AND_INDEPENDENT_SAVED_REVIEW"
assert len(fold_receipt["folds"]) == 2
print("新主线：两折真实65-session滑动，读取保存证据；旧v1 TopK另列。")
print("本页没有 Data、Feature、fit、predict 或账户调用。")


## 3. Feature ID 与依赖：由 registry 生成表

表从已合 Research 的 `stock_ml_v1.json` 自动生成，逐项核对本次保存 ModelRelease 的 selection / 版本 / 列顺序。稀疏 ID 按语义分组，不重新编号；公式、依赖与 lookback 继续由公共 catalog 和 Core 合同定义。

| 分组 | ID / 名称 | 公式或含义 | 依赖字段 | Lookback / session | 语义版本 | 本模型 |
|---|---|---|---|---|---|---|
| MOM | MOM010 / ret1 | adjusted_close[t] / adjusted_close[t-1] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 2 | 1.0.0 | 选用 |
| MOM | MOM020 / ret5 | adjusted_close[t] / adjusted_close[t-5] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 6 | 1.0.0 | 选用 |
| MOM | MOM030 / ret20 | adjusted_close[t] / adjusted_close[t-20] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 21 | 1.0.0 | 选用 |
| VOL | VOL010 / intraday_range | (high[t] - low[t]) / close[t] | market_daily.high, market_daily.low, market_daily.close, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| PRC | PRC010 / close_to_open | close[t] / open[t] - 1 | market_daily.close, market_daily.open, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| LIQ | LIQ010 / amount_relative_5 | amount_cny[t] / rolling_mean(amount_cny, window=5, inclusive_current=true) | market_daily.amount_cny | 5 | 1.0.0 | 选用 |

[固定 registry 源](https://github.com/sinnergarden/axiom-research/blob/4e2e0e1a5d382fc33f30ccec658976ff649141b1/src/axiom_research/catalogs/stock_ml_v1.json)。本轮没有重新执行 Feature Core，也没有增加特征或优化参数。

### 用公共 catalog 生成同一张表

下面是按本次六列 selection 生成表的用法展示，放在 Markdown 中，不随默认只读 cell 执行；它只读取 catalog，不运行 Feature Core。

```python
from axiom_research import load_feature_catalog

catalog = load_feature_catalog()
selection = [{"id": fid, "semantic_version": "1.0.0"} for fid in
             ("MOM010", "MOM020", "MOM030", "VOL010", "PRC010", "LIQ010")]
print("| 分组 | ID / 名称 | 公式或含义 | 依赖字段 | Lookback / session | 语义版本 | 本模型 |")
print("|---|---|---|---|---|---|---|")
for f in catalog.select(selection):
    fields = (f["group"], f'{f["id"]} / {f["name"]}', f["formula"],
              ", ".join(f["dependencies"]), str(f["lookback"]),
              f["semantic_version"], "选用")
    print("| " + " | ".join(fields) + " |")
```


## 4. Data / Qlib / FeatureBuild：保存事实，不改旧时钟

Data 负责固定 Snapshot/PIT/cutoff 事实与 Qlib 格式导出；Research 按 catalog 编译 FeaturePlan，由既有 Core 执行。跨仓合同复用，不再造 Feature 执行器。

本次 fold 从完整 Feature 父件选择完整日期截面，并保留原 `knowledge_cutoff=20:30` 和依赖 availability。文件字节 digest 与解析内容 ref 分别验证；旧带空格 JSON 合法且保持原样。加载仍完整解析父证明，不是 streaming，日期 slice 也不替代完整来源核验。

Data PR15/18/40 的审批门槛仍按父任务记录未解决；本稿仅链接状态，不收入其未批准实现，也不通过 Docs 合并绕过审批。


## 5. 成熟标签与三阶段时钟

目标仍是实际日历中 `s+1` 开盘至 `s+5` 收盘的五 session 收益，截面 zscore 由原归一化 Core 算子完成，ddof=0、排除成员/缺值等政策不变。fit 只能读取当时成熟的有限 target；未来 outcome 仅用于事后 IC/RankIC，不进入训练。

| 时钟 | 本有界 profile | 必须满足的关系 |
|---|---|---|
| Feature knowledge / available | 原 Feature 20:30；依赖 available 原值 | available ≤ Feature knowledge ≤ inference |
| fit knowledge / 模拟模型可用 | fit 当日 20:30 → 声明 20:45 | fit < model available < inference |
| inference / prediction publication | 预测 Feature 日期 21:00 | prediction knowledge = available，同 aware instant |

20:45 / 21:00 是声明的历史模拟时钟；本机耗时不能证明当时已完成训练。每个 OOS trade 日严格使用前一实际 session 的 Feature/预测。

两折各有 4 个未成熟 Feature 日期，完整 314 ID 留在排除证据中：`LABEL_NOT_MATURE=1,256`；成熟截面的非成员排除为 `NOT_MEMBER=854`。不缩短65日期窗口、不填缺值、不把声明窗口当全部训练键。


## 6. Dataset 与模型：两个真实移动窗口

公开入口 `build_stock_ml_fold_from_saved_inputs` 按固定实际 calendar，在 fit 前选 **65 个 Feature sessions**，只用截至 fit 的成熟有效 target。原 LightGBM 参数与共享后端保持不变：100 trees、seed=42、单线程，无搜索或 early stopping。

| Fold / fit | 65-session Feature 窗口 | 实际成熟窗口 / sessions | 训练行 | OOS trade 日 | 预测行 / valid |
|---|---|---|---:|---|---:|
| Feb5 / Feb2 | 2023-11-02—2024-02-01 | Nov2—Jan26 / 61 | 18,300 | Feb5—8 | 1,256 / 1,200 |
| Feb19 / Feb8 | 2023-11-08—2024-02-07 | Nov8—Feb1 / 61 | 18,300 | Feb19—23 | 1,570 / 1,500 |

窗口真实移动4个sessions。每折保存 Feature/Label slice、Dataset、native booster、ModelRelease v2、Prediction v2 与原 Signal evidence；definition 绑定所有原父件与实现来源。

| 首次构建测量 | Fold Feb5 | Fold Feb19 |
|---|---:|---:|
| builder 完整父件验证 | 4.147 s | 4.264 s |
| 三个矩阵转换合计 | 0.00534 s | 0.00589 s |
| fit / inference | 0.05672 / 0.00225 s | 0.05331 / 0.00282 s |
| 文件发布 | 0.110 s | 0.118 s |
| staged public loader | 4.900 s | 5.253 s |
| 首次 builder 总时间 | **10.178 s** | **9.846 s** |

内部阶段属于总时间，不再与它相加。两折完整构建/自验恢复流程50.78 s、峰值RSS1.87 GiB；含首次训练前失败累计62.23 s，监控结束累计文件40.27 MiB。独立复审额外只读载入两折，不包含在上述业务时钟内。

原同配置 helper 的13.14 s首建 /4.95 s HIT作为历史控制保留；它不是本次滑动 API，也不替代本表。

### 实际怎样调用一次滑动训练

下面是本次已经执行的公共 builder 用法，作为 Markdown 代码展示，**默认不执行**。`saved_inputs_dir` 指向 owner 保存的两份 `input-manifest.json`；示例目录名是占位，不随教程附带私人父件。每份 manifest 固定 Snapshot、calendar、314 ID、六列 selection，并以绝对父件路径、文件 digest 与逻辑 ref 绑定 scope、Feature / proof、raw / normalized training labels 及事后 evaluation labels。父件路径必须保留，不能只替换路径或使用 `latest`。

```python
import json
from pathlib import Path
from axiom_research import build_stock_ml_fold_from_saved_inputs

def train_two_saved_folds(saved_inputs_dir, output_dir):
    saved_inputs_dir, output_dir = Path(saved_inputs_dir), Path(output_dir)
    saved = []
    for first, last, fit in (
        ("2024-02-05", "2024-02-08", "2024-02-02"),
        ("2024-02-19", "2024-02-23", "2024-02-08"),
    ):
        manifest = json.loads(
            (saved_inputs_dir / first / "input-manifest.json").read_text())
        calendar = manifest["calendar"]  # 固定真实交易日；不按工作日猜测
        trades = [d for d in calendar if first <= d <= last]
        prediction_dates = [calendar[calendar.index(d) - 1] for d in trades]
        fold_spec = {
            "contract_version": "stock_ml_fold_spec_v1",
            "training_window": {"unit": "feature_sessions", "length": 65,
                                "end": "previous_fit_session"},
            "fit_session": fit,
            "fit_cutoff": fit + "T20:30:00+08:00",
            "simulated_model_available_at": fit + "T20:45:00+08:00",
            "oos_trade_sessions": trades,
            "inference_cutoff_by_session":
                {d: d + "T21:00:00+08:00" for d in prediction_dates},
            "evaluation_cutoff": "2024-02-29T20:30:00+08:00",
        }
        metrics = {}
        fold = build_stock_ml_fold_from_saved_inputs(
            manifest, fold_spec=fold_spec,
            destination=output_dir / first / "folds", metrics=metrics)
        saved.append({"fold_path": str(fold.path), "fold_ref": fold.identity,
                      "model_path": str(fold.path / "model.json"),
                      "booster_path": str(fold.path / "booster.txt"),
                      "signal_path": str(fold.path / "predictions.json"),
                      "signal_ref": fold.predictions()["signal_run_ref"]})
    return saved

# 另行授权训练时才显式调用；阅读此教程不调用：
# saved = train_two_saved_folds("saved-inputs", "saved-folds")
```

builder 内部仍复用既有 native LightGBM 共享后端：100 trees、seed=42、单线程；成熟性筛选来自原保存 Label，不把最后4个未成熟日期拿来训练。返回路径下还保存 Feature/Label slice、Dataset、Signal evidence 和文件 manifest。同一定义已存在时完整验证后 HIT，不重新 fit/predict；改输入或实现会形成独立定义。这里的 Prediction v2 仍只允许 Engine 中立校验，不能送入账户。

[`examples/weekly_ml_pilot.py`](../examples/weekly_ml_pilot.py) 是原 Nov1—Jan31 **固定日期窗口历史控制**，保留用于理解旧证据；新65-session滑动入口是上述公共 builder，本教程不维护另一套训练后端。


In [ ]:
def read_saved_fold(saved_fold_path):
    from axiom_research import load_stock_ml_fold
    saved = load_stock_ml_fold(saved_fold_path)
    return saved.to_dict(), saved.model(), saved.predictions()


# 该公开 loader 已在 owner / 独立复审中调用。本页不重复解析大 proof，只读其冻结回执。
for fold in fold_receipt["folds"]:
    print(fold["first_trade_session"], "Feature窗口", fold["training_window"],
          "成熟窗口", fold["actual_training_window"], "训练", fold["training_rows"],
          "预测/valid", fold["prediction_rows"], fold["valid_predictions"])
    print("saved fold", fold["fold_ref"], "Signal", fold["signal_run_ref"])


## 7. 保存 Signal：完整网格，分数与账户准入分开

新件为 `stock_prediction_run_v2`：`signal_stage=prediction_raw`、`score_semantics=forward_5_session_cs_zscore_prediction`、无量纲。分数是归一化 target 的预测，不是收益百分比。完整 union 的每日期314行都保留 member/valid/invalid_reason；非成员等无效行 score=null。

逐行保留原 Feature knowledge/availability，增加声明模型可用时钟，预测 knowledge=available=21:00。保存输入、模型与 fold_spec refs 固定，旧 v1 不改。两折均通过公共 `validate_stock_predictions`，独立加载 saved booster 的预测与原保存分数逐值相等。

**这只完成 v2 中立准入。planner / Runtime 明确拒绝 v2 账户消费。** 以下账户章节使用原 January v1 Signal，不把这两份新预测转换成 v1 或绑定旧账户。


## 8. Engine 唯一账户 / Evaluation：同 Signal 换组合

**同一旧 January v1 保存 Signal 的 Top3/Top5 对照已通过。** 两份独立账户与原 EvaluationReport v2 均已保存（评价v2与Prediction v2是不同合同），五个调仓日目标均不同。Data、供应商、Feature、fit、predict 调用全部为 0；完整输入与评价引用在本地回执逐项核对。

| 实际调仓日 | Top5 选中数 | Top3 选中数 | 目标组合核对 |
|---|---:|---:|---|
| 2024-01-02 | 5 | 3 | 不同 |
| 2024-01-08 | 5 | 3 | 不同 |
| 2024-01-15 | 5 | 3 | 不同 |
| 2024-01-22 | 5 | 3 | 不同 |
| 2024-01-29 | 5 | 3 | 不同 |

| 固定不变 | 实际变化 | 保存结果 |
|---|---|---|
| 原 January Signal / MarketReplay / 初始资金 / profile / 费用 / T+1 / lot / 执行范围与来源限制 | 仅 portfolio_policy.top_k：5 → 3 | 独立 run_id、完整账户与评价；原 Signal 和旧结果不重写 |
| Data / 供应商 / Feature / fit / predict | 均为 0 次 | 上游未导入、未调用；实际 Runtime / Evaluation 输出保留 |

TopK 合同已由 Docs PR17 审准，Engine PR8 已合。实际绑定已审源码 `f6d93a1`；`stock_portfolio_policy(top_k=k, execution_universe=原执行集合)` 返回现有请求政策，替换此字段后仍沿 `BacktestRequest.from_dict` / `run_backtest`。所有显式 k 使用 Core /2，旧 /1 Top5 保存件仍可读取。准确执行与统计定义见 [Trade](../docs/design/04_axiom_trade.md#61-有界股票日线-top5)。本例沿用固定 v2 Evaluation 输入，未混入后来新增的评价版本。

首次进程同时保留旧账户/评价、新账户及其展开字典，评价触发 RSS guard，失败记录保留。恢复将账户与评价分为独立进程：先核验已保存 Top5，再仅补 Top5 评价、Top3 账户和 Top3 评价，**没有重复 Top5 账户**。每阶段释放不用的对象并保留完整 proof，恢复 106.65 s、峰值 4.78 GiB，全部进程已退出。这处理本样本的生命周期峰值，尚不能证明多年 loader 内存有界。

下面两个函数对应已验收的阶段调用；必须在绑定已审 Engine 来源的受控新进程中分别调用。默认只定义函数并读取回执，已完成阶段不重复执行。


In [ ]:
def make_topk_account(saved_run_path, top_k, new_destination):
    import gc
    from axiom_engine.runtime import (load_backtest_run, stock_portfolio_policy,
        BacktestRequest, run_backtest, save_backtest_run)

    output = Path(new_destination)
    output.mkdir(parents=True, exist_ok=False)
    baseline = load_backtest_run(saved_run_path)
    plan = baseline.to_dict()["plan"]
    del baseline
    gc.collect()
    plan["portfolio_policy"] = stock_portfolio_policy(
        top_k=top_k, execution_universe=plan["execution_universe"])
    request = BacktestRequest.from_dict(plan)
    del plan
    gc.collect()
    run = run_backtest(request)
    del request
    gc.collect()
    save_backtest_run(run, output / "backtest.json")
    return output / "backtest.json"


def evaluate_saved_topk(saved_run_path, saved_reference_evaluation, new_destination):
    import gc
    from axiom_engine.runtime import (load_backtest_run, load_backtest_evaluation,
        BenchmarkSeries, DividendScope, EvaluationSpec, evaluate_backtest,
        save_backtest_evaluation)

    output = Path(new_destination)
    output.mkdir(parents=True, exist_ok=False)
    report = load_backtest_evaluation(saved_reference_evaluation).to_dict()
    benchmark = BenchmarkSeries.from_dict(report["benchmark_input"])
    scope = None if report["dividend_scope"] is None else DividendScope.from_dict(report["dividend_scope"])
    spec = EvaluationSpec.from_dict(report["spec"])
    del report
    gc.collect()
    run = load_backtest_run(saved_run_path)
    evaluation = evaluate_backtest(run, benchmark=benchmark, spec=spec, dividend_scope=scope)
    del run, benchmark, scope, spec
    gc.collect()
    save_backtest_evaluation(evaluation, output / "evaluation.json")
    return output / "evaluation.json"


# 历史 v1 Signal 对照的调用；不支持新 Prediction v2。本页不调用业务函数。
print("TopK 对照状态：", receipt["topk"]["status"])
for result in receipt["topk"]["outputs"]:
    print("Top", result["top_k"], "决策", result["decision_count"], "成交", result["fills"])
print("目标不同的调仓日：", ", ".join(receipt["topk"]["different_target_sessions"]))
print("上游实际调用：", receipt["topk"]["upstream_calls"])
print("账户调用 / 评价尝试 / 完整评价：",
      receipt["cumulative_counts"]["account"],
      receipt["cumulative_counts"]["evaluation_attempt"],
      receipt["cumulative_counts"]["completed_evaluation"])


## 9. UI：保存值进入同一个网页入口

UI Reader 校验 owner 保存件后生成投影；已导出 HTML 打开和切页不执行 Data、模型或账户。Pages 入口由既有发布流程维护，本教程不部署 UI。

旧版用户产品验收 FAILED 保留为历史。整改新版已于2026-10-05上线，父任务源码、实际截图与线上发布身份核验通过，owner线上交互QA通过；用户复验待进行。当前上线来源为 UI `4bdd8f0` / merge `969145d`，live gh-pages `8ec244d`。本教程新增 ML 教学公开候选仍由原 UI owner 从唯一 Notebook 生成，父审后再发布；工程试点通过不代替用户复验。

## 10. 重跑边界与已有测量：每种成本分开看

| 改动 / 动作 | 应复用 | 实际需要执行 / 当前限制 |
|---|---|---|
| 只打开已导出 HTML | 所有 owner 结果 | 仅浏览器；不跑 Reader / train / account |
| 加载保存模型 / Signal / run | 原结果内容 | 解析、hash/ref/证明核验；不是零成本 |
| 完全相同 builder 目标 | 全实验 | 公共调用命中后 Data / Core / fit / predict 为 0；仍完整加载核验 |
| 同一已准入 v1 独立预测 Signal 改组合 | Data / Feature / Model / Signal | Core 组合 / Runtime / Evaluation；同 Signal Top3/Top5 本轮通过，上游 0 次 |
| 改 label / 模型语义 | 未受影响 Feature / 原事实 | 新 Dataset / model / Signal；旧 helper 限同 config；新 saved-fold 绑定独立窗口/时钟与全部父件 |
| 改 Feature 公式 / 字段、Snapshot、PIT/cutoff | 仅实际未受影响阶段 | 新语义链；新增字段或导出 query 变化需新 Qlib view |
| 非语义 Python 修改 | 旧保存件仍可读 | wholeimplementation 可能过粗 MISS；不能报告为语义缓存能力 |

| 既有真实测量，未在本稿重跑 | 观察值 | 可用于解释的范围 |
|---|---|---|
| Data Qlib 314 ID × 56 sessions 冷准备（旧 exporter） | 37.72 s；峰值 5.43 GiB | 独立格式准备；不含模型 / Feature / 账户，新 Data PR39 无生产耗时 / RSS 实测 |
| 有界 monthly 两折 | 282.34 s；峰值 4.09 GiB | February 完整 15 日，March 仅 3/21 日；含额外等价核验，不是 weekly 或全年 |
| 其中 LightGBM fit / infer | fit 61.94 / 42.44 ms；infer 7.79 / 1.58 ms | 单阶段；矩阵/label/来源证明/序列化另算 |
| Engine 一组完整保存账户 + 评价加载 | 27.34 → 14.90 s，下降 45.5% | Reader GROUP 单样本；不是整个网页导出/浏览器速度 |
| 新 Data 导出 manifest 次数 | 冷 5→1；warm 每次重新核验 1 | 算法调用数，不能称 5 倍提速 |

OS 页缓存未清空，单阶段时间有嵌套，不能相加为另一个总时间；没有全年吞吐保证。


## 11. 保存后只读：验证、HIT 与重跑边界

| 两折实际自验 | Fold Feb5 | Fold Feb19 | 新 fit / predict |
|---|---:|---:|---:|
| fresh public load | 4.753 s | 4.761 s | 0 / 0 |
| Engine neutral v2 validator | 0.032 s | 0.041 s | 0 / 0 |
| saved booster 独立预测 | 0.183 s | 0.185 s | 0 / 1（每折） |
| 完全相同定义 HIT | 9.496 s | 9.780 s | 0 / 0 |

HIT 与 loader 仍完整核验原父件：owner 流程总共10次，每次父件投影验证4.15–4.69 s；独立review再只读载入两次。旧 spaced proof 的逻辑 digest fallback每次约0.010 s，并保留其序列化峰值阶段。保存件可读不要求当前实现版本；实现整体digest发生变化时也不能为了展示再build一个新身份。

首次真实尝试只完成2个 Data outcome读取与1个Feb1截面归一化，然后在旧JSON格式校验处停止，fit=0。修复经父亲审后复用保存准备件；恢复Data/norm/Feature/供应商/账户均0，fit/infer各2，另2次独立saved predict。16个原/准备父件hash、mtime、大小不变。

旧 weekly成熟控制保持Nov1—Jan31日期固定，曾以不同cutoff得到62/65个成熟session；它的66分钟情景只保留历史，不再称真实滑动估计。公开saved-fold已实现；通用schedule/多年份/streaming恢复平台仍未实现。


In [ ]:
# 已保存两折的校验结果；不调用 builder、不重训、不重推理。
for fold in fold_receipt["folds"]:
    assert fold["neutral_validation"] == "PASS"
    assert fold["independent_predict"] == "EXACT_EQUAL"
    assert fold["cache_fit_predict_calls"] == [0, 0]
    assert fold["engine_runtime"] == "UNSUPPORTED_V2"
    print(fold["first_trade_session"], "中立校验PASS / 独立预测相等 / HIT零fit,predict")
print("恢复业务调用：", fold_receipt["execution_counts"])
print("含首次准备的累计调用：", fold_receipt["cumulative_execution_counts"])


## 12. 五年计划：真实滑窗样本的条件 warm 估计

实际保存日历在2021-01-01—2025-12-31有**1,212 sessions /256个有交易日的ISO周**，平均每周4.734375日；全周休市不计fold。这是日历计数，不证明五年股票来源或历史成员完备。

**前提：每周fit所需原Feature、raw/normalized Label父件已准备就绪，314 ID、65-date窗口、原参数与本次证明规模近似不变。** 新观测来自真正移动的两折，已包含builder完整父件核验、矩阵、fit/infer、写件和staged loader。

| 条件成本 | 从保存秒数外推 | 256周参考 |
|---|---|---:|
| 已准备父件的saved-fold首次构建 | `256 × mean(10.178307,9.846465)/60` | **42.72 min** |
| 两个观测最小/最大情景 | `256 × [min,max]/60` | **[42.01,43.43] min** |
| 若逐周照做本次全部自验（fresh load/HIT/独立预测等） | `256/2 × 50.501216/60` | 107.74 min |
| 单列native fit+infer（已含于首建，不再加） | `256 × mean(0.05672+0.00225,0.05331+0.00282)` | 14.73 s |

这是保存父件充分准备、固定证明规模的场景，区间不是统计置信界或多年承诺。全部自验情景不等于必要生产步骤；两折平均4.5个OOS日与五年日历均值不同，预测侧成本未作线性容量保证。

完整计划仍必须另计：

`T_total = T_唯一Data/Qlib/Feature冷准备 + T_每fit_cutoff的Label准备/归一化 + T_saved_fold_warm[42.01,43.43]min + ΔT_来源证明与成员范围扩张 + T_账户及评价 + T_导出展示`

| 未测成本 | 本轮证据 / 限制 |
|---|---|
| 五年唯一日期Data/Qlib/Feature冷准备 | 旧37.72 s的314 ID×56日Qlib不含全冷链；本轮未重跑 |
| 每个未来cutoff的Label准备/归一化 | 本两折复用已有归一化父件，只补一个Feb1截面；不能把新cutoff全链填0 |
| 扩张后的完整proof解析 / I/O / 内存 | 当前每次读取完整父件；不存在已证明的streaming或多年常驻内存界 |
| 五年账户/评价 | 新v2预测还未账户准入；旧v1 TopK短月不证明该成本 |
| 导出 / UI / 浏览器 | 与owner计算分开；本次只读HTML渲染不代表多年报告成本 |

因此**完整五年总分钟数和有限上界均未知**，没有启动全年/五年执行。

### 历史固定窗口估计

此前Nov1—Jan31固定日期控制的65.15–66.87 min（中心66.01）包含其共享outcome/norm与原证明流程。它不是新滑窗的最终warm估计，成本边界不同，也不能把66→42.72称为提速。本稿保留原场景与失败历史供核对。


## 13. 工程附录：历史尝试与当前边界

初始授权为同配置 1 次 fit/infer + 第二次 HIT、两周各 1 次 fit/infer、公共 label outcome 查询最多 8 次、供应商 0；账户最多 2 次。一个重进程，硬 RSS 6 GiB、自有 PID 在 5.5 GiB 停止；每阶段 120 s、业务时钟 480 s、累计新产物 1 GiB。

两次 weekly 失败均发生在查询证据序列化，各只读 1 个价格 batch、未训练。修复后两周共享事后 outcome 来源，成功流程 6 次读取；累计 8，失败目录保留。业务并非一次无错完成，缓存首建和 weekly 也分别计时。

TopK 首次评价采样峰值 **5.82 GiB**，触发 own-PID guard 后退出，Top5 账户已完整保存、Top3 尚未开始；该失败记录及产物没有覆盖。审准的恢复仅补缺少阶段，四个独立进程共用额外 480 s，不重跑 Top5，仍沿用相同 RSS/累计文件上限。实际恢复 **106.65 s / 4.78 GiB**，累计文件约 **674 MiB**，全部进程退出。

| 历史固定窗/TopK累计（不含新滑动fold） | 实际 |
|---|---:|
| Data 读取 / 供应商请求 | 8 / 0 |
| Feature Core 执行 | 0 |
| 固定配置 + weekly fit / predict | 3 / 3 |
| TopK 独立账户 | 2 |
| TopK 评价尝试 / 完整评价 | 3 / 2 |

仅 TopK 的 Data/供应商/Feature/fit/predict 均为 0；label/norm 使用既有 Core 算子，不能把“Feature Core 0”解释为所有 Core 计算 0。Notebook 当前四个默认代码单元只读取保存回执、定义函数，零业务调用；私有输出副本与全部原输入保留，本公开 source 不包含私人产物。

缓存 key 与 proof 有界加载的最小候选见 [Research §8.4.1](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)：实际计划决定上下文；两日期 shard 只为本次配置；顶层调用重验、调用内 proof blob 验一次后共享并释放。落盘本身不证明 streaming，正式 rolling/resume 尚未实现。冷 Qlib 准备沿用历史证据，未重跑。

新真实saved-fold单独预算480s/100MiB：首次准备失败11.45s、恢复50.78s，均未触发预算；恢复阶段Data/norm为0；含首次准备累计Data=2、归一化=1，旧失败与原保存件保留。新fold累计2个build fit/infer、2个独立saved predict，不与上表历史3次训练混算。默认四代码单元只读两份冻结回执、定义显式函数，0业务调用。


### 缓存与常驻内存：当前风险

| 现状与风险 | 建议目标 | 小例如何验 |
|---|---|---|
| Research builder 把 Research Python、Data Python 与 Core Python 的整体实现 digest 纳入身份；非语义代码改动也可能 MISS，旧helper限同配置；新saved-fold绑定独立窗口/时钟及全父件 | 后续按实际计算依赖与语义版本拆分身份；保留完整实现来源用于审计。变更前先冻结哪些修改应失效，避免默默宽松复用 | 同输入加载旧保存件仍通过；当前 MISS 如实报告。先列失效矩阵，教程不直接改生产 cache |
| 长历史 Feature / proof 大对象可能一直驻留；两折小例未证明多年内存有界 | 按完整日期截面分片持久化、逐片校验并释放；固定 parent/ref 图与原子 checkpoint | 先完整一周、再代表范围测 RSS / 文件字节 / 中断恢复；未通过不扩全年 |

目标方向已接受，saved-fold已冻结并完成两折；语义cache分层与有界proof新API仍待冻结；两折不证明完整平台或 streaming 已实现。
四项候选的准确边界统一见 [Research §8.4.1](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)：阶段依赖身份保留实际计算实现与完整审计；首轮按两个完整日期分片、原 proof blob 独立保存并每次核验。它们尚未实施，本教学不先大改缓存或 loader。


### SysQ：借经验，明确已有差异

| 已读证据中的 SysQ 做法 | Axiom 当前实现与取舍 |
|---|---|
| Feature per-window write-through；命中后读原窗口 frame，再训练 | 已有固定单折保存输入复用。本轮已有公开saved-input fold入口与真实滑窗；通用rolling缓存/schedule平台仍未实现 |
| SignalStore / LabelStore / 实验索引；信号分析与 signal-driven backtest 分开 | 保存 Signal 后交 Engine 唯一账户；只读 UI 使用 owner 保存件 |
| 已读 research usecase 的成熟门槛与 lineage 检查仍列 TBD | 当前逐行成熟时钟、输入 digest、完整 universe / invalid、分数阶段已明确保留；不沿用工作日 fallback、填零或当前成员替代历史 |

此表沿用 [此前固定 SysQ 对照](../docs/design/05_axiom_research.md#stock-qlib-lightgbm-minimal) 中的 usecase / generator / training 证据，不是新一轮全仓审计或 SysQ 当前全量评价。

## 14. 三个小用例：从正式输入到保存结果

这部分沿“Feature/Label → Model → 原始 Prediction → 组合 Signal → 信号评价 → 策略账户”展开。下面的 CaseC 单元直接调用正式入口，真实数据引用尚待交接，本轮只核对语法和轻量单元。CaseA 的可配置 TrainSpec、CaseB 的通用组合继续留在待接区，完成各自接口后再执行。

| 编号与目的 | 当前能力 | 待接项 | 输入与输出 |
|---|---|---|---|
| 1. CaseC：四个周度 fold 与 Top5，优先验收 | matrix_v2 保存准备、fold v3、OOS 评价冻结和 Runtime v7 均有正式入口 | 完整 Feature、四折时钟、回放与评价输入的真实引用；本轮未运行 | 同一六列 Feature 和 5D Label → 四份模型/原始预测、信号评价、50 万元账户与账户评价 |
| 2. CaseA：同输入比较两个训练配置 | CaseC 可验证准备复用与同配置 fold HIT | builder 仍固定 LightGBM 参数，窄 TrainSpec 待实现 | 同一 prepared 矩阵及两个真实配置 → 两份模型/预测和阶段成本 |
| 3. CaseB：保存预测组合后统一评价 | 保存预测可经 OOS frozen-input 路径，使用独立 Raw evaluationLabel | 正式组合及其保存件的评价准入；按年汇总与分组收益入口 | 原始预测、固定权重、同一实际可用 Label → 保存组合与共同样本比较 |

从本节单独执行，使用 Python 3.11 及以上的 kernel，先安装 owner 交接的 Data、Research、Engine 环境；第 1–13 节的历史回执无须加载。所有新增输出保存在外部验收根目录，公开 Notebook 保持空输出。6、158、300 列复用同一正式列式和窗口入口；当前固定六列，扩列所需正式定义由 Research 交接。Data Qlib view 保存事实，Research prepared 矩阵保存派生训练输入，两者沿用同一事实来源。

**输入目录。** 设置 `AXIOM_M1_ROOT` 为实际验收根目录，在其中放入 `case-inputs.json`。下表和 JSON 只说明位置；每份输入由原 owner 保存并核验，日期、版本、时钟和预算采用交接值。保存件内部已有的绝对定位按原协议保留，迁移时由原 owner 交接可验证的完整闭包。

```json
{
  "data_root": "data",
  "feature_inputs": "inputs/features-six",
  "fold_specs": "inputs/four-weekly-folds.json",
  "preparation_options": "inputs/preparation-options.json",
  "batch_limits": "inputs/batch-limits.json",
  "evaluation_scope": "inputs/signal-evaluation-scope.json",
  "raw_evaluation_label": null,
  "runtime_request": "inputs/top5-request-v7.json",
  "runtime_limits": "inputs/runtime-limits.json",
  "block_sessions": 5,
  "benchmark": "inputs/csi300-benchmark.json",
  "dividend_scope": "inputs/dividend-scope-v3.json"
}
```

`features-six` 是已完成的 `stock_feature_inputs_v3` 目录，采用 `matrix_v2`，包含原来源引用。`fold_specs` 是四个有序、OOS 互不重叠的正式 spec，覆盖约四个交易周和各自训练窗口。`preparation_options` 精确提供 `row_block_sessions`、`column_block`、`maximum_resident_bytes`、`normalization_backend`；后者固定为 `core_cs_batch_v1`。`batch_limits` 提供 `maximum_source_bytes` 和 `maximum_matrix_bytes`。Runtime 请求、预算及账户评价输入分别见 [Trade §6.5](../docs/design/04_axiom_trade.md#stock-bounded-v7-proposal) 与 [Trade §11.2](../docs/design/04_axiom_trade.md#long-history-evaluation)。

`raw_evaluation_label` 留空时，冻结入口取同一 batch 的原 OOS Raw 目标；也可填写一个小型 descriptor 文件的相对位置，显式提供共同评价 Label。scope 固定评价日期、证券、完整日历和评价 cutoff，须覆盖目标成熟所需日期。训练 target 使用 5D 横截面标准化目标，评价使用原收益 Label，各自保存定义。

下一个单元只加载配置、小型 Feature 索引和正式 catalog，并建立本次计时文件。只有后续单元会执行准备、训练或账户。准备及 fold 缓存保留在根目录 `cache/`，每次运行的记录和独立账户保存到新的 `runs/` 子目录。缺少输入会明确报错，待补齐后按单元继续。

In [ ]:
import json
import os
import resource
import sys
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

ROOT = Path(os.environ["AXIOM_M1_ROOT"]).expanduser().resolve(strict=True)

def case_path(relative):
    value = Path(relative)
    if value.is_absolute():
        raise ValueError("配置位置应相对 AXIOM_M1_ROOT")
    result = (ROOT / value).resolve()
    if not result.is_relative_to(ROOT):
        raise ValueError("配置位置超出验收根目录")
    return result

def read_small_json(path, maximum=16 * 1024**2):
    # 只读配置和小型索引；矩阵及来源闭包交给正式 reader。
    with Path(path).open("rb") as saved:
        raw = saved.read(maximum + 1)
    if len(raw) > maximum:
        raise ValueError("小型配置超过读取预算")
    return json.loads(raw)

cfg = read_small_json(ROOT / "case-inputs.json")
feature_path = case_path(cfg["feature_inputs"])
feature_index = read_small_json(feature_path / "index.json")
if feature_index["contract_version"] != "stock_feature_inputs_v3" or feature_index["status"] != "COMPLETE":
    raise ValueError("待交接完成的 matrix_v2 / stock_feature_inputs_v3 输入")
feature_spec = feature_index["definition"]["spec"]
from axiom_research import load_feature_catalog
catalog = load_feature_catalog()
selected = catalog.select(feature_spec["feature_selection"])
if [item["id"] for item in selected] != feature_spec["ordered_features"] or len(selected) != 6:
    raise ValueError("首个 CaseC 需要交接的正式六列选择")
for item in selected:
    print({key: item[key] for key in ("id", "name", "formula", "dependencies", "lookback", "semantic_version")})
del feature_index, feature_spec, selected, catalog

run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_ROOT = ROOT / "runs" / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=False)
timing_path = RUN_ROOT / "timings.jsonl"

@contextmanager
def measure(stage, api_metrics=None):
    # 计时探针只记录调用；业务由下面各单元直接执行。
    before = resource.getrusage(resource.RUSAGE_SELF)
    started = time.perf_counter()
    record = {"stage": stage, "status": "RETURNED"}
    try:
        yield
    except BaseException as error:
        record.update(status="ERROR", error_type=type(error).__name__)
        raise
    finally:
        after = resource.getrusage(resource.RUSAGE_SELF)
        record.update(elapsed_seconds=time.perf_counter() - started,
            rss_process_peak_bytes=int(after.ru_maxrss * (1 if sys.platform == "darwin" else 1024)),
            input_block_ops_delta=after.ru_inblock - before.ru_inblock,
            output_block_ops_delta=after.ru_oublock - before.ru_oublock,
            api_metrics={} if api_metrics is None else api_metrics)
        with timing_path.open("a", encoding="utf-8") as output:
            output.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        print(record)


### 14.1 CaseC：准备一次，保存四折，再运行 Top5

1. **目的与输入。** 固定六列 Feature、实际五交易日 Label、四折日历和时钟，检查成熟训练、原始 OOS 预测、唯一 Runtime 账户及资源成本。完整 Feature 输入须覆盖训练窗；正在生成的产物完成并交接后才能使用。
2. **准备与预期。** 下方只调用一次 prepare，生成共享 prepared view 和四折 selectors，Data 保持 `cache_bytes=0`。首次精确定义为 MISS；已有同定义保存件可 HIT，两种结果均记录实际计数。HIT 的 Data/Core/fit/predict/account 调用应为零。prepare 不重新 build Feature，完成后沿正式实现释放当前 Data 实例缓存。
3. **要看什么。** 检查准备耗时、逻辑 Data 调用、Core 调用、实际读/hash 字节和工作集峰值。物理分区计数尚未由公共 Data API 暴露时，保留正式报告中的空值和原因。这里的 Data 内存缓存与磁盘 prepared HIT 分别解释。

In [ ]:
from axiom_data import Data
from axiom_research import prepare_stock_ml_batch_inputs

fold_specs = read_small_json(case_path(cfg["fold_specs"]))
if type(fold_specs) is not list or len(fold_specs) != 4:
    raise ValueError("CaseC 需要四个正式周度 fold spec")
preparation_options = read_small_json(case_path(cfg["preparation_options"]))
batch_limits = read_small_json(case_path(cfg["batch_limits"]))
data = Data(case_path(cfg["data_root"]), cache_bytes=0, max_readers=4)
prepare_metrics = {}
with measure("C.prepare", prepare_metrics):
    prepared = prepare_stock_ml_batch_inputs(data, feature_inputs=feature_path,
        fold_specs=fold_specs, destination=ROOT / "cache" / "prepared",
        preparation_options=preparation_options, metrics=prepare_metrics)
del data
assert prepared["contract_version"] == "stock_ml_batch_inputs_v2"
assert len(prepared["folds"]) == 4
assert prepare_metrics["feature_core_calls"] == prepare_metrics["train_calls"] == prepare_metrics["predict_calls"] == 0


**四折保存与复用。** 本单元只初始化一个 batch，并在同一生命周期内构建四折、逐折检查同定义 HIT、冻结 OOS 评价输入，结束时释放矩阵。每折只投影当前训练窗和声明列；模型参数沿用当前固定 LightGBM。每份保存结果分别保留 Dataset、Model 和原始 Prediction；小型 metadata 展示成熟训练行数、训练键引用、排除原因及 OOS 交易区间。

第一次每折可能新 fit/predict，也可能已有保存件 HIT，实际次数由 builder 记录。随后同配置重试应 HIT 且零 fit/predict；此检查仍会核验保存件和当前投影，成本单列。batch metrics 记录共同索引构建、descriptor 校验、hash/decode、投影次数与字节。中断后重新使用同定义入口可重用完成的 Research fold，Runtime 账户从新请求完整运行。

冻结沿正式 OOS-only 路径，保留输入模型/fold 来源和原评价 Label。这里复用 batch 的共享来源表；独立完整来源审计留给整体复审。教程没有调用私有投影方法，也没有重新写证券/日期 join。

In [ ]:
from axiom_research import (load_stock_ml_batch_inputs,
    build_stock_ml_fold_from_saved_inputs, save_stock_signal_evaluation_inputs)

signal_scope = read_small_json(case_path(cfg["evaluation_scope"]))
raw_override = None if cfg["raw_evaluation_label"] is None else read_small_json(case_path(cfg["raw_evaluation_label"]))
fold_records, prediction_inputs = [], []
batch_metrics = {}
with measure("C.batch.lifecycle", batch_metrics):
    with measure("C.batch.admission"):
        batch = load_stock_ml_batch_inputs(prepared, limits=batch_limits)
    with batch:
        for number, item in enumerate(prepared["folds"], 1):
            metrics = {}
            with measure(f"C.fold.{number}.build", metrics):
                fold = build_stock_ml_fold_from_saved_inputs(item["input_manifest"],
                    fold_spec=item["fold_spec"], destination=ROOT / "cache" / "folds",
                    metrics=metrics, batch=batch)
            meta = read_small_json(fold.path / "fold.json")
            files = read_small_json(fold.path / "manifest.json")["files"]
            dataset = read_small_json(fold.path / "dataset.json")
            record = {key: meta[key] for key in ("fold_ref", "model_ref", "feature_ref", "signal_run_ref")}
            record.update(fold_spec_ref=meta["definition"]["fold_spec_ref"],
                training_rows=dataset["training_row_count"],
                training_keys_digest=dataset["training_keys_digest"],
                excluded=dataset["excluded"],
                oos_trade_sessions=item["fold_spec"]["oos_trade_sessions"])
            fold_records.append(record)
            prediction_inputs.append({"path": str(fold.path / "predictions.json"),
                "file_digest": files["predictions.json"], "signal_run_ref": meta["signal_run_ref"]})
            print(record)
            hit_metrics = {}
            with measure(f"C.fold.{number}.same_config_hit", hit_metrics):
                hit = build_stock_ml_fold_from_saved_inputs(item["input_manifest"],
                    fold_spec=item["fold_spec"], destination=ROOT / "cache" / "folds",
                    metrics=hit_metrics, batch=batch)
            assert hit.reused and hit.path == fold.path
            assert hit_metrics["cache_hit"] and hit_metrics["train_calls"] == hit_metrics["predict_calls"] == 0
            del fold, hit, meta, dataset, files
        with measure("C.signal.freeze"):
            signal_input_ref = save_stock_signal_evaluation_inputs(
                {"baseline_5d": prediction_inputs}, raw_label_input=raw_override,
                scope=signal_scope, destination=ROOT / "cache" / "signal-inputs", batch=batch)
        batch_metrics.update(batch.metrics)
        assert batch_metrics["common_key_index_builds"] == 1


**独立信号评价。** 下方仅消费已冻结的输入，保存整体 IC、RankIC、非年化 ICIR、共同有效样本和各信号自然覆盖，然后原定义重试验证评价 HIT。HIT 复用统计结果，冻结文件仍会核验；该阶段的正常返回与完整用例通过分别记录。当前统计入口尚未输出按年汇总和分组 Label 收益，补齐后按预测日期所属年份下钻。长 horizon 的重叠样本与短窗口 ICIR需谨慎解释；Top5 扣费收益另从账户评价读取。

In [ ]:
from axiom_research import evaluate_stock_signal_inputs

with measure("C.signal.evaluate"):
    signal_reports = evaluate_stock_signal_inputs(signal_input_ref,
        scope=signal_scope, destination=ROOT / "cache" / "signal-evaluations")
for name, saved in signal_reports.items():
    report = saved.to_dict()
    print({"signal": name, "status": report["status"], "summary": report["summary"],
        "common_pairs": report["coverage"]["valid_pair_count"],
        "natural_pairs": report["coverage"]["native"]["valid_pair_count"],
        "excluded": report["coverage"]["excluded_counts"], "saved_report": str(saved.path)})
with measure("C.signal.same_definition_hit"):
    signal_hit = evaluate_stock_signal_inputs(signal_input_ref,
        scope=signal_scope, destination=ROOT / "cache" / "signal-evaluations")
assert all(saved.reused for saved in signal_hit.values())
del report, signal_hit, signal_reports


**唯一 Runtime 账户。** Engine owner 交接的 v7 请求须逐项引用上面四折保存预测、原 model metadata、fold spec、执行行情与 profile；请求会绑定现金分红、成员可见时间、成交规则和费用。下方核对引用与初始账户，再让 `run_stock_backtest` 执行其正式两遍扫描及同一 planner/simulator/ledger。输入扫描的完整核验已包含在回测入口中。

初始现金为 50 万元，保存单位为 CNY 分，所以填 `50000000`；持仓为空，策略为周度 Top5。账户结果写入本次新目录，当前 v7 的任意崩溃恢复尚待实现。输出要看订单、成交、ledger 和扣费 NAV 的相同提交序号，另记录费用、未成交数量、源审计计数与 result parts。账户耗时包含正式输入准入和执行，保存单独计时。

In [ ]:
from axiom_engine.runtime import (BacktestRequest, StockInputSource, StockResultSink,
    stock_run_id, run_stock_backtest, save_backtest_run)

request_wire = read_small_json(case_path(cfg["runtime_request"]))
runtime_limits = read_small_json(case_path(cfg["runtime_limits"]))
assert request_wire["contract_version"] == "backtest_request_v7"
assert request_wire["initial_account"] == {"cash_minor": 50000000, "positions": {}}
assert request_wire["portfolio_policy"]["top_k"] == 5
keys = ("fold_ref", "fold_spec_ref", "model_ref", "feature_ref", "signal_run_ref")
assert [tuple(frame[key] for key in keys) for frame in request_wire["prediction_input"]["frames"]] == [tuple(row[key] for key in keys) for row in fold_records]
request = BacktestRequest.from_dict(request_wire)
runtime_path = RUN_ROOT / "runtime" / "run.json"
runtime_metrics = {}
with measure("C.runtime.run", runtime_metrics):
    runtime_run = run_stock_backtest(request, source=StockInputSource(),
        sink=StockResultSink(RUN_ROOT / "runtime" / "parts", run_id=stock_run_id(request_wire)),
        block_sessions=cfg["block_sessions"], limits=runtime_limits)
    runtime_wire = runtime_run.to_dict()
    runtime_metrics.update(runtime_wire["metrics"])
with measure("C.runtime.save"):
    save_backtest_run(runtime_run, runtime_path)
print({"status": runtime_wire["status"], "metrics": runtime_wire["metrics"],
    "source_counts": runtime_wire["source_audit"]["counts"],
    "result_parts": len(runtime_wire["result_parts"])})
assert runtime_wire["status"] == "COMPLETE"
del runtime_run, request, request_wire, runtime_wire


**从保存账户做评价。** 下方只读取 v7 保存结果、分片和小型 profile，再消费独立保存的 CSI300 benchmark 与分红范围。原生行情和训练父件沿完整回测准入核验，此处复用正式保存结果投影。基准须覆盖账户前一 anchor 和全部 NAV 日期，并绑定执行侧 Snapshot；分红范围按 v3 绑定该账户的 account events。

查看扣费总收益、最大回撤、最终 NAV、费用、基准对齐及评价的完整/部分状态。三到四周窗口的 CAGR会保留跨度不足状态；本单元的基础评价无 Sharpe 输出，后续沿正式分析评价补齐。结果是否完整取决于原输入覆盖，正常返回本身不表示整个 CaseC 通过。评价保存与实际耗时均在本次目录中。

In [ ]:
from axiom_engine.runtime import (load_stock_backtest_projection, BenchmarkSeries,
    DividendScope, long_history_evaluation_spec, evaluate_backtest, save_backtest_evaluation)

with measure("C.account.saved_projection"):
    projection = load_stock_backtest_projection(runtime_path,
        artifact_reader=lambda ref: ref["manifest_uri"], limits=runtime_limits)
with measure("C.account.evaluate"):
    benchmark = BenchmarkSeries.from_dict(read_small_json(case_path(cfg["benchmark"])))
    dividend_scope = DividendScope.from_dict(read_small_json(case_path(cfg["dividend_scope"])))
    account_report = evaluate_backtest(projection, benchmark=benchmark,
        spec=long_history_evaluation_spec(), dividend_scope=dividend_scope)
with measure("C.account.save"):
    save_backtest_evaluation(account_report, RUN_ROOT / "runtime" / "evaluation.json")
account = account_report.to_dict()
print({"status": account["status"], "period_metrics": account["period_metrics"],
    "benchmark_status": account["benchmark"]["status"],
    "last_nav": projection.rows["nav"][-1], "runtime_metrics": projection.wire["metrics"]})
del projection, benchmark, dividend_scope, account_report, account
print("本次阶段记录：", timing_path)


### 14.2 CaseA：同输入比较两份 TrainSpec，待接口

1. **目的。** 固定事实、Feature、训练 target、窗口与成熟训练键，只改变一个训练参数，比较模型/预测与真实成本。
2. **当前能力。** CaseC 已接同一 prepared view 和 batch 内的 fold HIT 单元，能够记录准备、训练和预测的实际次数与耗时。当前正式 builder 固定 LightGBM 参数。
3. **差距与待接输入。** Research 需要交接窄 TrainSpec 的准确公共调用、两份真实配置，以及模型身份如何绑定配置。当前没有两份配置的训练对照结果；这里保留接口 TODO，接入时沿用 CaseC 的 batch 生命周期和计时位置。
4. **执行与输出要求。** 对同一准备输入依次首建配置一、原配置 HIT、配置二；保存两份 Model 和原始 OOS Prediction，并核对相同训练键及预测范围。兼容的 Feature/Label/prepared 层应复用；配置变化触发新的 fit/predict。分别列首建、HIT、只改配置的次数、阶段耗时和投影成本，模型优劣由独立评价观察。

### 14.3 CaseB：固定加权组合，同一 evaluationLabel，待组合

1. **目的与输入。** 用两组真实保存原始预测和显式固定权重生成可复用组合 Signal，再把两组原预测和组合放到同一实际可用的五交易日评价 Label 上比较。训练 target 与 evaluationLabel 各自固定，60/180 训练及 Y40/Y180待真实目标具备后接入。
2. **当前能力。** CaseC 展示了准确的 `save_stock_signal_evaluation_inputs(..., batch=batch)` 与 `evaluate_stock_signal_inputs(...)` 调用，已支持矩阵 OOS 输入和显式 Raw Label override；共同有效样本、各信号自然覆盖与整体 IC/RankIC 可保存复用。更换 evaluationLabel 时保留模型和预测，只重建对应冻结输入与评价。
3. **差距。** 通用保存组合入口仍待实现；当前矩阵准入接受正式 fold 的 `predictions.json`，组合保存件的准入也需原 owner 交接。因此本节暂不放组合执行单元。另待按年汇总、分组标签收益的正式统计入口，Notebook 沿同一入口展示结果。
4. **执行与输出要求。** 正式组合按同证券/session精确 join，缺输入保持 invalid；权重固定，rank/zscore 只在显式指定时调用，参考 cohort 沿预测生成时冻结。保留全部 model/fold 来源和包含变换、发布延迟的 availability。冻结比较组的评价定义/版本、日期、证券和共同成熟 mask，并另报自然覆盖；整体、按预测年份的 IC/RankIC/非年化 ICIR及分组 Label 收益分别保存。预测或 Label 缺失不会触发重新标准化、填零、ffill 或重分权。IC 提升与 Top5 扣费 NAV分别从信号和账户结果判断。

**待接引用清单。** 两组真实原始预测 descriptor、固定权重与显式变换定义、正式组合保存引用及准入合同、共同 Raw evaluationLabel descriptor、冻结 scope。准确调用由原 owner 提供并经独立审查后补入，继续使用 CaseC 的共享 batch与计时探针。

### 14.4 结果记录与验收顺序

计时位置为 `$AXIOM_M1_ROOT/runs/<本次运行>/timings.jsonl`。每条记录包含阶段实际耗时、正式 API metrics、进程从启动起的峰值 RSS以及本阶段的块 IO 操作数变化。嵌套的 batch 生命周期耗时包含 admission、fold 和冻结子阶段，合计时只取一层。块操作数保留其原单位；实际物理字节只使用 owner 暴露的计数，尚无接口的项目注明待测。后台进程和 notebook kernel 的历史峰值会影响 RSS，最终效率验收需同环境独立进程记录。

| 观察层 | 应看到的复用与重要指标 | 记录位置 |
|---|---|---|
| Data / Feature | Data cache0；保存 Feature 只读；公共物理分区计数待 owner 暴露 | `C.prepare` 的实际 Data/Core/来源 metrics |
| Prepared | 本流程 prepare一次；已有完全同定义结果可 HIT | `C.prepare`，`cache/prepared/.../batch.json` |
| Batch / fold | 一个 batch；当前窗投影；同配置 HIT 零 fit/predict，仍核验保存件 | `C.batch.lifecycle`、各 `C.fold.*`，`cache/folds/...` |
| 信号评价 | 冻结 OOS 输入；相同定义统计 HIT，仍核验冻结文件 | `C.signal.*`，`cache/signal-inputs/`、`cache/signal-evaluations/` |
| Runtime / 账户评价 | 新账户完整执行；评价复用保存结果，记录 NAV、回撤、费用、缺失状态 | `C.runtime.*`、`C.account.*`，本次 `runtime/` |

先逐单元接 CaseC 的四折小窗口，自查成熟 mask、键、值、null、成员、时钟、排名边界和引用，再检查实际准备、投影、IO与RSS，固定补丁提交父审；整体运行及独立来源复审另行安排。本轮新增单元无输出，未声称 Notebook、训练或账户验收通过。

小例之后沿同一实现做 M1 多年正确性与性能验收，CaseA/B 随原接口逐项补齐。M1 全部完成后才进入 M1.5 的 Mac → WSL 主开发环境迁移，日期随完成情况确定。M2 检验 CAGR 25%、最大回撤 25%、Sharpe 1–1.2的研究目标，M3 用隔离 shadow，平台回测统一以 50 万元起步，M4全自动交易后置。项目顺序见 [Research §11](../docs/design/05_axiom_research.md#11-最小实施路径)。